# Berlin Emergency Response
**INTRODUCTION**

---

**Zweck:** Das Projekt erklären und zeigen, was darin steckt: Fragestellung, Datenquellen, Architektur, Arbeitsweise.  
**Input:** keiner, die Zahlen unten kommen live aus `data/berlin_emergency.duckdb`.  
**Output:** Orientierung für alle weiteren Notebooks (`01`–`04`).

## Facts

| Feld | Wert |
|---|---|
| **Frage** | Welche Faktoren hängen mit der Antwortzeit des Berliner Rettungsdienstes zusammen, und wo und wann wird die Hilfsfrist verfehlt? |
| **Stakeholder** | Interessierte Öffentlichkeit, Fachpublikum aus Daten und Rettungswesen |
| **Methode** | Explorative und beschreibende Analyse auf einer dbt-Pipeline (DuckDB), Zusammenhänge statt Ursachen |
| **Hauptdatenquelle** | Open Data der Berliner Feuerwehr (`BF-Open-Data`, CC BY 4.0) |
| **Zielgrößen** | Antwortzeit in Sekunden, Anteil Einsätze innerhalb der Hilfsfrist (offiziell und eigene Näherung) |
| **Nicht Teil des Projekts** | Bewertung von Einsatzkräften, Kausalaussagen, Prognosemodelle |

## Research Question

> Welche Faktoren hängen mit der Antwortzeit des Berliner Rettungsdienstes zusammen (Zeit, Ort, Einsatzart und Dringlichkeit, Sondersituationen, gegebenenfalls Wetter), und wo und wann wird die Hilfsfrist verfehlt?

Wir nutzen die öffentlich zugänglichen Daten der Berliner Feuerwehr und reichern sie schrittweise mit weiteren Daten an. Gezeigt werden **Zusammenhänge, keine Ursachen**: Antwortzeiten hängen an Standortdichte, Verkehr, Bebauung und Einsatzaufkommen, nicht an der Leistung einzelner Teams.

### Dimensions

| Dimension | Frage | Notebook | Datenbasis |
|---|---|---|---|
| Zeit | Wie verändern sich Antwortzeiten über Jahre, Saison und Wochentag? | `03_analysis_time` | Einsätze, Tagesreihe |
| Ort | Wo sind Antwortzeiten länger, wo wird die Hilfsfrist verfehlt? | `03_analysis_geo` | Bezirke (Einsätze), Regionen (Jahresaggregat) |
| Einsatzart und Mittel | Hängt die Zeit an Einsatztyp, Dringlichkeit, Einsatzmitteln? | `03_analysis_mission_type` | Einsätze |
| Sonderereignisse | Pandemie, Silvester, Strukturbrüche in der Definition | `03_analysis_special_events` | Einsätze, Tagesreihe, `seed_events` |
| Wetter, Feiertage | Gibt es Effekte von Witterung, Feiertagen und Ferien? | `03_analysis_weather` | Anreicherung (geplant) |

### Hypotheses

Vermutungen, die wir in den Notebooks prüfen, nicht Annahmen:

* Vermutung: Antwortzeiten sind in dünn bebauten Außenbezirken länger als im Zentrum.
* Vermutung: Die Hilfsfrist wird in einsatzstarken Zeiten (Sommer, Wochenenden) häufiger verfehlt.
* Vermutung: Der Anstieg der offiziellen Hilfsfrist-Quote von 2024 auf 2025 hängt an einer geänderten Definition von "kritisch", nicht an schnelleren Einsätzen.
* Vermutung: Sondersituationen (Pandemiejahre, Jahreswechsel) zeigen sich in Einsatzzahlen stärker als in Antwortzeiten.

### Limits of the Data

* Keine Uhrzeit und keine Einsatzkoordinaten, nur Tag und Bezirk (12). "Wo" heißt: Bezirk im Zeitverlauf, Region pro Jahr.
* Regionaldaten (Bezirksregion, Planungsraum, Prognoseraum) nur als Jahresaggregat und nur ab 2024.
* Bei einem Teil der Einsätze fehlt die Antwortzeit; Einsätze haben keine ID, Duplikate sind nicht erkennbar.
* Die Grundgesamtheit "kritischer" Einsätze ändert sich zwischen 2024 und 2025 (siehe `seed_events`); Jahresvergleiche der offiziellen Quote sind eingeschränkt.

## Data Sources

Alle Einsatzdaten stammen aus dem Repository [`Berliner-Feuerwehr/BF-Open-Data`](https://github.com/Berliner-Feuerwehr/BF-Open-Data) (CC BY 4.0). Upstream hat keinen Tagesendpunkt: täglich wird die Datei des laufenden Jahres neu geladen, Vorjahre einmalig (`scripts/ingest.py`).

### Overview

| Quelle | Granularität | Inhalt | Warum wir sie nutzen | Was für uns wichtig ist |
|---|---|---|---|---|
| **Mission_Data** (`mission_data_set_open_data_<Jahr>.csv`) | ein Einsatz, 2018 bis heute | Tag, Einsatzart, Dispatch-Code und Stufe (A–E, O), Bezirk, Antwortzeit, Einsatzmittel (`units_*`), Notarzt, First Responder | einzige Quelle auf Einsatzebene, trägt Zeitverlauf und Abhängigkeiten | `response_time`, `mission_type`, `dispatchcode_criticality`, `mission_location_district`, `units_first_type` |
| **Daily_Data** (`BFw_mission_data_daily.csv`) | Stadt je Tag, ab 2018 | Einsatzzahlen nach Kategorie, Statistik der Antwortzeiten, Zählungen `mission_count_rd1` bis `rd5` | Gegenprobe zu Mission_Data, Tagesreihe für Sonderereignisse | `mission_count_*`, `response_time_ems_critical_*` |
| **Regional_Data** (`<Jahr>/BFw_{planning_room,district_area,prediction_area}_data_<Jahr>.csv`) | LOR-Region je Jahr, ab 2024 | Einsatzzahlen, Antwortzeit-Statistik, **Hilfsfrist: berechnet und erreicht** | einzige Quelle mit offizieller Hilfsfrist-Quote und geografischer Tiefe | `*_timegoal_computed`, `*_timegoal_reached`, drei LOR-Ebenen mit stabilen IDs |
| **Turnout_Times** (`crew_turnout_times_<Jahr>_q<N>.csv`, `_current`) | Wache je Quartal | Alarmierungen und mittlere Ausrückdauer für RTW, NEF, LHF, DLK, ELW | Anteil der Ausrückzeit an der Gesamtzeit, Blick auf Wachen | `mittlere_ausrueckedauer_*`, `wache_name` |
| **Dispatchcodes** (`Dispatchcodes/*.xlsx`) | Code | Zuordnung Dispatch-Code zu Ressourcenanforderung | Namen der Codes; derzeit nutzen wir die Paare (Code, Name) aus Mission_Data als Seed | `seed_dispatch_codes` |
| **LOR-Polygone** (Geoportal Berlin, WFS `su_lor`, Stand 01.01.2021) | 3 Ebenen | Flächen der Planungsräume, Bezirksregionen, Prognoseräume | Karte; IDs stimmen mit den BF-Regionen überein | CC BY 3.0 DE, Quellenvermerk Pflicht |
| **Wetter, Feiertage und Ferien** (geplant) | Tag | Witterung (Open-Meteo), Kalender | Anreicherung Stufe 1 | folgt nach dem ersten EDA-Durchlauf |

Nicht genutzt: `Daily_Data/BFw_call_data_*` (Notrufzahlen) und `KV_Data`. Die Disclaimer-Datei weist darauf hin, dass Notrufzahlen bei Rückfallbetrieb unvollständig sind.

### Response Time and Time Goal

* **Antwortzeit** (`response_time`): Zeit von der Einsatzanlage bis zum Eintreffen des ersten Fahrzeugs, in Sekunden. BF dokumentiert im Upstream-README, dass für Notruf und Disposition zusätzlich zwei Minuten angesetzt werden.
* **Hilfsfrist, offiziell:** Anteil der kritischen Rettungsdiensteinsätze, bei denen die Frist erreicht wurde (`timegoal_reached` geteilt durch `timegoal_computed`), nur in den Regionaldaten.
* **Hilfsfrist, eigene Näherung:** Anteil der Rettungsdiensteinsätze der gewählten Dispatch-Stufen mit Antwortzeit unter einer Schwelle (dbt-Var `timegoal_seconds`, Vermutung). Sie reproduziert die offizielle Quote nicht in allen Jahren; das klärt `03_analysis_time`.

## Architecture

### Medallion Layers

```
BF-Open-Data (GitHub)   LOR-Polygone (WFS)
        │                      │
        ▼                      │
scripts/ingest.py              │   idempotent, harter Fehler statt Fallback
        │
        ▼
BRONZE   raw_*                 unveränderte Rohdaten (VARCHAR), _partition, _loaded_at
        │  dbt
        ▼
SILVER   stg_* → int_*         typisiert, benannt, Bezirke und Flags, LOR-Hierarchie
        │
        ▼
GOLD     fct_* / dim_*         analysefertige Marts (table, incremental)
        │  scripts/export_parquet.py
        ▼
SERVING  app_data/*.parquet    nur das, was die App liest (+ insights.json aus 04_insights)
```

### dbt Components

| Komponente | Wo | Wie wir sie anwenden | Warum |
|---|---|---|---|
| **Sources + Freshness** | `models/staging/sources.yml` | sechs Raw-Tabellen; Freshness über `_loaded_at` für Einsätze und Tagesreihe (warn nach 2, error nach 4 Tagen) | trennt Ingestion von Transformation, meldet veraltete Daten |
| **Seeds** | `seeds/` | `seed_districts` (Bezirkscode und Name), `seed_dispatch_codes`, `seed_events` (belegte Ereignisse mit Quelle) | kleine, versionierte Referenzdaten statt Hardcodes im SQL |
| **Staging** (view) | `models/staging/` | 1:1 zur Quelle: umbenennen, typisieren, keine Geschäftslogik; Macros `to_int`, `stg_regional` | Typen und Namen an genau einer Stelle |
| **Macros, Jinja, Vars** | `macros/`, `dbt_project.yml` | `is_within_timegoal`, `safe_pct`, Var `timegoal_seconds`; Schleifen über Spaltenlisten | Schwelle zentral änderbar, kein Copy-Paste |
| **Intermediate** (view) | `models/intermediate/` | Bezirk-Join, Einsatzgruppe, Stufe, Hilfsfrist-Flag; die drei Regional-Ebenen vereint, Hierarchie aus den IDs | Geschäftslogik getrennt von Typisierung und Marts |
| **Marts** (table, incremental) | `models/marts/` | `fct_timegoal_region_yearly`, `fct_missions_daily_district` (incremental: `delete+insert` auf `mission_date`, Filter über `loaded_at`), `fct_missions_daily_citywide`, `fct_turnout_times_quarterly`, `dim_district`, `dim_region` | große Faktentabelle nicht täglich komplett neu bauen |
| **Tests** | `schema.yml`, `tests/singular/` | generisch (`unique`, `not_null`, `relationships`, `accepted_values`), `dbt_utils` (Kombination, Wertebereich, Ausdruck), singulär (Abgleich Regional gegen Einsätze, keine Zukunftsdaten, Ebenen stimmig) | Datenfehler fallen vor der App auf |
| **Snapshots, Docs, Exposures, Contracts, Groups, Versions** | S5 | folgt | SCD2-Historie der Regionaldaten, Lineage, Governance der Marts |

### Automation

Heute laufen die Schritte über `make ingest`, `make dbt-build`, `make export`. In S4 übernimmt GitHub Actions täglich: Ingestion, Freshness, Build, Export, Commit der App-Daten.

## Workflow

### Notebook Map

| Notebook | Aufgabe | Output |
|---|---|---|
| `00_introduction` | Fragestellung, Quellen, Architektur | diese Orientierung |
| `01_exploration_<quelle>` | je Quelle durchleuchten (`wgnd.inspect`) und ein Fazit: Inhalt, Nutzen, Grenzen | Befunde je Quelle |
| `02_preparation` | die Pipeline im Detail: Bereinigung, Zusammenführung, Aufbereitung, Automatisierung mit dbt | nachvollziehbarer Weg Bronze zu Gold |
| `03_analysis_<dimension>` | gezielte Untersuchungen auf den Gold-Tabellen | belegte Muster |
| `04_insights` | Erkenntnisse zusammenführen, Schlussfolgerungen, Empfehlungen, neue Charts | Grundlage der App (`app_data/insights.json`) |

### Conventions

| Variable | Zustand |
|---|---|
| `con` | schreibgeschützte DuckDB-Verbindung (`read_only=True`) |
| `df_raw` | Auszug aus einer `raw_*`-Tabelle, unverändert |
| `df_edit` | bereinigt oder aggregiert, immer mit `.copy()` aus `df_raw` |
| `df_final` | Ergebnis für Charts und Insights |

Charts nach `public/img/`, Auswertungen für die App nach `app_data/`. Pfade über `PATHS`.

## Setup

In [1]:
from berlin_emergency_response.notebook import *
import duckdb

setup_plotting()
con = duckdb.connect(str(PATHS["db"]), read_only=True)

✓  wgnd theme activated (matplotlib · seaborn)

### Rows per Layer

In [2]:
df_tables = con.sql("select table_name from information_schema.tables where table_schema = 'main'").df()

def layer_of(name: str) -> str:
    for prefix, layer in [('seed_', '0 seed'), ('raw_', '1 bronze'), ('stg_', '2 silver'), ('int_', '2 silver'), ('fct_', '3 gold'), ('dim_', '3 gold')]:
        if name.startswith(prefix):
            return layer
    return ''

df_tables['layer'] = df_tables['table_name'].map(layer_of)
df_layers = df_tables[df_tables['layer'] != ''].sort_values(['layer', 'table_name']).copy()
df_layers['rows'] = [con.sql(f'select count(*) from {t}').fetchone()[0] for t in df_layers['table_name']]
show_df(df_layers[['layer', 'table_name', 'rows']].reset_index(drop=True))

,layer,table_name,rows
0,0 seed,seed_dispatch_codes,69
1,0 seed,seed_districts,12
2,0 seed,seed_events,2
3,1 bronze,raw_daily_mission_data,3195
4,1 bronze,raw_mission_data,4461202
5,1 bronze,raw_regional_district_area,429
6,1 bronze,raw_regional_planning_room,1626
7,1 bronze,raw_regional_prediction_area,174
8,1 bronze,raw_turnout_times,2675
9,2 silver,int_missions_enriched,4461202
